# 🚀 YOLOv8 Training - Google Colab Version

## Ventajas de Colab vs Local (RTX 3050)

| Aspecto | RTX 3050 Local | Colab Free (T4) | Colab Pro (V100/A100) |
|---------|----------------|-----------------|----------------------|
| VRAM | 4GB | 16GB | 16-40GB |
| Batch size | 8 | **16-32** | **32-64** |
| Velocidad | Base | Similar | 2-4x más rápido |

## Instrucciones
1. Sube este notebook a Google Colab
2. Cambia el runtime a GPU: `Runtime > Change runtime type > GPU`
3. Sube tu dataset (o descárgalo desde Google Drive)
4. Ejecuta las celdas

---

## 1. 🔧 Configurar Entorno Colab

In [ ]:
# =============================================================================
# Install Dependencies (Colab doesn't have ultralytics pre-installed)
# =============================================================================

!pip install -q ultralytics

# Verify installation
import ultralytics
ultralytics.checks()

In [ ]:
# =============================================================================
# Check GPU Type
# =============================================================================

import torch

print("="*60)
print("INFORMACIÓN DE GPU EN COLAB")
print("="*60)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    
    print(f"GPU detectada: {gpu_name}")
    print(f"Memoria VRAM: {gpu_mem:.1f} GB")
    
    # Recommend batch size based on GPU
    if "T4" in gpu_name:
        print("\n📊 GPU T4 detectada (Colab Free)")
        print("   - Batch recomendado: 16")
        print("   - Tiempo estimado (100 épocas): ~5h")
        RECOMMENDED_BATCH = 16
    elif "V100" in gpu_name:
        print("\n📊 GPU V100 detectada (Colab Pro)")
        print("   - Batch recomendado: 32")
        print("   - Tiempo estimado (100 épocas): ~3h")
        RECOMMENDED_BATCH = 32
    elif "A100" in gpu_name:
        print("\n📊 GPU A100 detectada (Colab Pro+)")
        print("   - Batch recomendado: 64")
        print("   - Tiempo estimado (100 épocas): ~1.5h")
        RECOMMENDED_BATCH = 64
    else:
        print(f"\n📊 GPU desconocida, usando batch=16")
        RECOMMENDED_BATCH = 16
else:
    print("❌ NO HAY GPU!")
    print("\nVe a: Runtime > Change runtime type > GPU")
    RECOMMENDED_BATCH = 8

## 2. 📁 Subir Dataset

### Opción A: Subir desde tu PC (más lento)

In [ ]:
# =============================================================================
# Option A: Upload dataset as ZIP
# =============================================================================
# Run this cell and upload dataset_yolov8.zip from your PC
# =============================================================================

from google.colab import files
import zipfile
import os

print("📤 Sube el archivo dataset_yolov8.zip...")
print("(O salta esta celda y usa Google Drive)")

# Uncomment to upload
# uploaded = files.upload()

# # Unzip
# with zipfile.ZipFile('dataset_yolov8.zip', 'r') as zip_ref:
#     zip_ref.extractall('/content/')
# print("✅ Dataset extraído en /content/dataset_yolov8/")

### Opción B: Usar Google Drive (recomendado)

In [ ]:
# =============================================================================
# Option B: Mount Google Drive
# =============================================================================
# 1. Upload dataset_yolov8.zip to your Google Drive
# 2. Run this cell to mount Drive
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

# Adjust path if your dataset is in a different location
DRIVE_DATASET_PATH = "/content/drive/MyDrive/dataset_yolov8.zip"

import os
import zipfile

if os.path.exists(DRIVE_DATASET_PATH):
    print("📦 Extrayendo dataset desde Drive...")
    with zipfile.ZipFile(DRIVE_DATASET_PATH, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("✅ Dataset extraído en /content/dataset_yolov8/")
else:
    print(f"❌ No se encontró: {DRIVE_DATASET_PATH}")
    print("\nSube dataset_yolov8.zip a tu Google Drive y ajusta la ruta.")

## 3. 🔧 Actualizar data.yaml

Las rutas del data.yaml deben apuntar a la ubicación en Colab.

In [ ]:
# =============================================================================
# Update data.yaml paths for Colab
# =============================================================================

import yaml

DATA_YAML_PATH = "/content/dataset_yolov8/data.yaml"

# Read current config
with open(DATA_YAML_PATH, 'r') as f:
    config = yaml.safe_load(f)

# Update paths for Colab
config['train'] = '/content/dataset_yolov8/train/images'
config['val'] = '/content/dataset_yolov8/valid/images'
config['test'] = '/content/dataset_yolov8/test/images'

# Write updated config
with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ data.yaml actualizado para Colab")
print(f"   - Train: {config['train']}")
print(f"   - Val: {config['val']}")
print(f"   - Clases: {config['nc']}")

## 4. ⚙️ Configuración Optimizada para Colab

In [ ]:
# =============================================================================
# Training Configuration - Optimized for Colab
# =============================================================================

# Model - you can use larger models in Colab due to more VRAM
MODEL_NAME = "yolov8s.pt"  # Can use yolov8m on V100/A100

# Training parameters
EPOCHS = 100
PATIENCE = 25

# Image size
IMAGE_SIZE = 640

# Batch size - use the recommended value for your GPU
# T4: 16, V100: 32, A100: 64
BATCH_SIZE = RECOMMENDED_BATCH if 'RECOMMENDED_BATCH' in dir() else 16

# Workers - Colab has 2 CPU cores
WORKERS = 2

# Learning rate
LR_INITIAL = 0.001

# Paths
DATA_YAML = "/content/dataset_yolov8/data.yaml"
EXPERIMENT_NAME = "logo_detection_colab"

print("="*60)
print("CONFIGURACIÓN PARA COLAB")
print("="*60)
print(f"Modelo: {MODEL_NAME}")
print(f"Épocas: {EPOCHS}")
print(f"Batch size: {BATCH_SIZE} (ajustado para tu GPU)")
print(f"Image size: {IMAGE_SIZE}")
print(f"Workers: {WORKERS}")
print("="*60)

## 5. 🏋️ Entrenamiento

In [ ]:
# =============================================================================
# Training
# =============================================================================

from ultralytics import YOLO
import time

print("Cargando modelo...")
model = YOLO(MODEL_NAME)

print("\n" + "="*60)
print("🚀 INICIANDO ENTRENAMIENTO EN COLAB")
print("="*60)
print(f"⏱️  Inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📦 Batch: {BATCH_SIZE}")
print("="*60 + "\n")

start_time = time.time()

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    patience=PATIENCE,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=0,
    workers=WORKERS,
    save=True,
    project="/content/runs/detect",
    name=EXPERIMENT_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer="AdamW",
    lr0=LR_INITIAL,
    lrf=0.1,
    warmup_epochs=5,
    mosaic=1.0,
    mixup=0.1,
    cache=True,  # Colab has enough RAM to cache images
    verbose=True,
    plots=True,
    seed=42,
)

training_time = time.time() - start_time
hours = int(training_time // 3600)
minutes = int((training_time % 3600) // 60)

print("\n" + "="*60)
print("✅ ENTRENAMIENTO COMPLETADO")
print("="*60)
print(f"⏱️  Tiempo total: {hours}h {minutes}m")

## 6. 📊 Resultados

In [ ]:
# =============================================================================
# Display Results
# =============================================================================

import pandas as pd
from IPython.display import Image, display

RESULTS_PATH = f"/content/runs/detect/{EXPERIMENT_NAME}"

# Metrics
df = pd.read_csv(f"{RESULTS_PATH}/results.csv")
df.columns = df.columns.str.strip()
last = df.iloc[-1]

print("="*60)
print("MÉTRICAS FINALES - COLAB")
print("="*60)
print(f"\nmAP@0.50:      {last.get('metrics/mAP50(B)', 0):.4f}")
print(f"mAP@0.50-0.95: {last.get('metrics/mAP50-95(B)', 0):.4f}")
print(f"Precision:     {last.get('metrics/precision(B)', 0):.4f}")
print(f"Recall:        {last.get('metrics/recall(B)', 0):.4f}")

# Curves
print("\n📊 Curvas de entrenamiento:")
display(Image(filename=f"{RESULTS_PATH}/results.png", width=1000))

## 7. 💾 Descargar Modelo Entrenado

In [ ]:
# =============================================================================
# Download trained model
# =============================================================================

from google.colab import files
import shutil

# Path to best model
BEST_MODEL = f"/content/runs/detect/{EXPERIMENT_NAME}/weights/best.pt"

# Copy to Drive for persistence
DRIVE_DEST = "/content/drive/MyDrive/models/best_colab.pt"
import os
os.makedirs(os.path.dirname(DRIVE_DEST), exist_ok=True)
shutil.copy(BEST_MODEL, DRIVE_DEST)
print(f"✅ Modelo guardado en Drive: {DRIVE_DEST}")

# Download to PC
print("\n📥 Descargando modelo a tu PC...")
files.download(BEST_MODEL)

## 📝 Comparación Local vs Colab

Después de ejecutar, actualiza esta tabla:

| Métrica | RTX 3050 Local | Colab (T4) | Colab (V100) |
|---------|----------------|------------|---------------|
| Tiempo | 2h 42m (50 épocas) | ??? | ??? |
| mAP@0.50 | 0.281 | ??? | ??? |
| Batch usado | 8 | 16 | 32 |